In [1]:

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge, Lasso, ElasticNet


# ==================================
# 1. LOAD DATASET
# ==================================

df = pd.read_csv("house_price_multicollinearity_100_rows.csv")

print("Dataset Shape:", df.shape)
print(df.head())


# ==================================
# 2. SEPARATE INPUT AND TARGET
# ==================================

X = df.drop("price_lakh", axis=1)

y = df["price_lakh"]


# ==================================
# 3. TRAIN TEST SPLIT
# ==================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)


# ==================================
# 4. RIDGE REGRESSION (L2)
# ==================================

ridge_model = make_pipeline(
    StandardScaler(),
    Ridge()
)

ridge_params = {
    'ridge__alpha': [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    ridge_model,
    ridge_params,
    cv=5,
    scoring='r2'
)

ridge_grid.fit(X_train, y_train)

print("\nRIDGE REGRESSION")
print("Best Parameters:", ridge_grid.best_params_)
print("R2 Score:", ridge_grid.score(X_test, y_test))


# ==================================
# 5. LASSO REGRESSION (L1)
# ==================================

lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(max_iter=10000)
)

lasso_params = {
    'lasso__alpha': [0.001, 0.01, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
    lasso_model,
    lasso_params,
    cv=5,
    scoring='r2'
)

lasso_grid.fit(X_train, y_train)

print("\nLASSO REGRESSION")
print("Best Parameters:", lasso_grid.best_params_)
print("R2 Score:", lasso_grid.score(X_test, y_test))


# ==================================
# 6. ELASTIC NET (L1 + L2)
# ==================================

elastic_model = make_pipeline(
    StandardScaler(),
    ElasticNet(max_iter=10000)
)

elastic_params = {
    'elasticnet__alpha': [0.01, 0.1, 1, 10],
    'elasticnet__l1_ratio': [0.2, 0.5, 0.8, 1]
}

elastic_grid = GridSearchCV(
    elastic_model,
    elastic_params,
    cv=5,
    scoring='r2'
)

elastic_grid.fit(X_train, y_train)

print("\nELASTIC NET REGRESSION")
print("Best Parameters:", elastic_grid.best_params_)
print("R2 Score:", elastic_grid.score(X_test, y_test))


# ==================================
# 7. COMPARE ALL THREE MODELS
# ==================================

results = {
    "Ridge": ridge_grid.score(X_test, y_test),
    "Lasso": lasso_grid.score(X_test, y_test),
    "Elastic Net": elastic_grid.score(X_test, y_test)
}

print("\nMODEL COMPARISON")

for name, score in results.items():
    print(f"{name}: {score:.4f}")

print("\nBest Model:", max(results, key=results.get))


Dataset Shape: (100, 7)
   house_size_sqft  house_size_sqm  bedrooms  bathrooms  age_years  \
0              905           84.10         2          3         24   
1             2480          230.40         5          3          3   
2             2206          204.95         4          2         22   
3             1709          158.79         2          1          2   
4             1696          157.52         2          2         15   

   distance_city_km  price_lakh  
0               3.2       37.81  
1              22.5       83.71  
2               1.7       81.37  
3               6.8       59.96  
4               4.4       63.17  
Training Data: (80, 6)
Testing Data: (20, 6)

RIDGE REGRESSION
Best Parameters: {'ridge__alpha': 0.1}
R2 Score: 0.9892088467807147

LASSO REGRESSION
Best Parameters: {'lasso__alpha': 0.001}
R2 Score: 0.989177042655752

ELASTIC NET REGRESSION
Best Parameters: {'elasticnet__alpha': 0.01, 'elasticnet__l1_ratio': 0.8}
R2 Score: 0.9892303682094727

MODEL